In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os.path as path
import utils
import glob

In [2]:
nuts_df = gpd.read_file(path.join(utils.raw_data_dir, "NUTS_RG_01M_2021_4326.shp"))

In [3]:
nuts_df = nuts_df.to_crs(epsg=3035)
nuts_df["area_km2"] = round(nuts_df.geometry.area / 1e6, 2)

In [4]:
population_data = pd.read_csv(path.join(utils.raw_data_dir, "estat_demo_r_pjangrp3.tsv"))
population_data = population_data[population_data["sex"] == "T"]
population_data = population_data[population_data["age"] == "TOTAL"]

In [5]:
last_col = population_data.columns[-1]

population_data[last_col] = population_data[last_col].apply(lambda x: [e for e in x.split("\t") if e != ": " ])
population_data["NUTS_ID"] = population_data[last_col].apply(lambda x: x[0])
population_data["population"] = population_data[last_col].apply(lambda x: x[-1])
population_data = population_data.drop(columns=[last_col, "sex", "age", "freq", "unit"])

In [6]:
nuts_df = pd.merge(nuts_df, population_data, on="NUTS_ID", how="outer")

In [7]:
nuts3_df = nuts_df[nuts_df["LEVL_CODE"] == 3].drop(columns=["LEVL_CODE", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE"])

In [8]:
crop_profile_files = glob.glob(path.join(utils.intermediate_data_dir, "nuts3_crop_profile", "*.geojson"))
crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files])

/var/folders/1_/j9jx95wx4zv29v6wh_ztx5_r0000gp/T/ipykernel_16753/1508858051.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  crop_profile_df = pd.concat([gpd.read_file(f) for f in crop_profile_files])


In [9]:
def calc_crop_area(in_dict):
    in_dict = eval(in_dict)
    out_dict = {}
    for k, v in in_dict.items():
        if k == 0:
            # default value for non-cropland pixels is 0
            continue
        crop_name = utils.cropland_type_dict[k]
        # NOTE: this next step is important and deserves explanation
        # the raster size of the cropland dataset is _precisely_ 10x10m for all pixels, as defined by the CRS
        # thus, each pixel has an area of 100m^2. We get the total area in m^2 by multiplying the pixel value by 100
        # to get from m^2->km^2 we need to divide by 1.000*1.000, i.e. 1.000.000
        # in other words, we divide by 10.000 or 1e4
        area_km = round(v * 1e-4, 2)
        out_dict[crop_name] = area_km
    return out_dict

crop_profile_df["cropland_km2_by_type"] = crop_profile_df["crop_profile"].apply(calc_crop_area)
crop_profile_df["cropland_km2"] = crop_profile_df["cropland_km2_by_type"].apply(lambda x: round(sum(x.values()), 2))


In [10]:
crop_profile_df = crop_profile_df[["NUTS_ID", "cropland_km2", "cropland_km2_by_type"]]
nuts3_df = pd.merge(nuts3_df, crop_profile_df, on="NUTS_ID", how="outer")
nuts3_df["cropland_area_percent"] = round(100 * nuts3_df["cropland_km2"] / nuts3_df["area_km2"], 2)

In [11]:
nuts3_drought_data = gpd.read_file(path.join(utils.intermediate_data_dir, "drought_days_nuts3.geojson"))
nuts3_drought_data = nuts3_drought_data[["NUTS_ID", 'median_drought_days', 'median_warning_days', 'median_alert_days',]]

In [12]:
nuts3_df = pd.merge(nuts3_df, nuts3_drought_data, on="NUTS_ID", how="outer")

In [13]:
#nuts3_df = nuts3_df.rename(columns={"CNTR_CODE_x": "CNTR_CODE", "NUTS_NAME_x": "NUTS_NAME", "NAME_LATN_x": "NAME_LATN", "geometry_x": "geometry",})
nuts3_df = nuts3_df[["NUTS_ID", "CNTR_CODE", "NUTS_NAME", 'median_drought_days', 'population', "area_km2", 'cropland_km2', 'cropland_area_percent', 'cropland_km2_by_type', "NAME_LATN",  'median_warning_days', 'median_alert_days', "geometry"]]
nuts3_df.set_index("NUTS_ID", drop=True, inplace=True)
nuts3_df

,CNTR_CODE,NUTS_NAME,median_drought_days,population,area_km2,cropland_km2,cropland_area_percent,cropland_km2_by_type,NAME_LATN,median_warning_days,median_alert_days,geometry
NUTS_ID,,,,,,,,,,,,
AL011,AL,Dibër,55.36,104624,2470.31,23.11,0.94,"{'Wheat': 1.96, 'Barley': 1.04, 'Maize': 10.11...",Dibër,50.48,5.59,"POLYGON ((5180040.081 2144853.889, 5179764.448..."
AL012,AL,Durrës,139.44,222999,772.54,144.52,18.71,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",Durrës,120.75,4.62,"POLYGON ((5139612.246 2104838.588, 5140583.791..."
AL013,AL,Kukës,98.97,60207,2391.45,8.18,0.34,"{'Wheat': 0.59, 'Barley': 0.1, 'Maize': 1.67, ...",Kukës,85.48,12.26,"POLYGON ((5154970.036 2212802.267, 5156310.157..."
AL014,AL,Lezhë,145.74,96384,1662.35,111.43,6.70,"{'Wheat': 8.56, 'Barley': 1.56, 'Maize': 21.55...",Lezhë,130.48,14.35,"POLYGON ((5166877.641 2159728.207, 5167535.432..."
AL015,AL,Shkodër,120.99,149496,3528.30,219.51,6.22,"{'Wheat': 5.83, 'Barley': 1.44, 'Maize': 35.0,...",Shkodër,100.06,12.12,"POLYGON ((5121233.536 2221719.441, 5120808.595..."
...,...,...,...,...,...,...,...,...,...,...,...,...
XK003,NaN,NaN,NaN,NaN,NaN,153.46,NaN,"{'Wheat': 30.48, 'Barley': 3.56, 'Maize': 48.2...",NaN,NaN,NaN,None
XK004,NaN,NaN,NaN,NaN,NaN,87.84,NaN,"{'Wheat': 20.66, 'Barley': 5.05, 'Maize': 10.5...",NaN,NaN,NaN,None
XK005,NaN,NaN,NaN,NaN,NaN,95.86,NaN,"{'Wheat': 8.84, 'Barley': 0.33, 'Maize': 18.27...",NaN,NaN,NaN,None


In [14]:
nuts3_df.to_file(path.join(utils.out_data_dir, "nuts3_stats.geojson"))

In [15]:
groups = nuts3_df.groupby("CNTR_CODE")
for ctr, df in groups:
    df.drop(columns=["geometry", "CNTR_CODE"]).to_excel(path.join(utils.out_data_dir, "nuts3_stats_by_country", f"{ctr}.xlsx"))

In [20]:
nuts3_df

,CNTR_CODE,NUTS_NAME,median_drought_days,population,area_km2,cropland_km2,cropland_area_percent,cropland_km2_by_type,NAME_LATN,median_warning_days,median_alert_days,geometry
NUTS_ID,,,,,,,,,,,,
AL011,AL,Dibër,55.36,104624,2470.31,23.11,0.94,"{'Wheat': 1.96, 'Barley': 1.04, 'Maize': 10.11...",Dibër,50.48,5.59,"POLYGON ((5180040.081 2144853.889, 5179764.448..."
AL012,AL,Durrës,139.44,222999,772.54,144.52,18.71,"{'Wheat': 19.74, 'Barley': 1.65, 'Maize': 28.9...",Durrës,120.75,4.62,"POLYGON ((5139612.246 2104838.588, 5140583.791..."
AL013,AL,Kukës,98.97,60207,2391.45,8.18,0.34,"{'Wheat': 0.59, 'Barley': 0.1, 'Maize': 1.67, ...",Kukës,85.48,12.26,"POLYGON ((5154970.036 2212802.267, 5156310.157..."
AL014,AL,Lezhë,145.74,96384,1662.35,111.43,6.70,"{'Wheat': 8.56, 'Barley': 1.56, 'Maize': 21.55...",Lezhë,130.48,14.35,"POLYGON ((5166877.641 2159728.207, 5167535.432..."
AL015,AL,Shkodër,120.99,149496,3528.30,219.51,6.22,"{'Wheat': 5.83, 'Barley': 1.44, 'Maize': 35.0,...",Shkodër,100.06,12.12,"POLYGON ((5121233.536 2221719.441, 5120808.595..."
...,...,...,...,...,...,...,...,...,...,...,...,...
XK003,NaN,NaN,NaN,NaN,NaN,153.46,NaN,"{'Wheat': 30.48, 'Barley': 3.56, 'Maize': 48.2...",NaN,NaN,NaN,None
XK004,NaN,NaN,NaN,NaN,NaN,87.84,NaN,"{'Wheat': 20.66, 'Barley': 5.05, 'Maize': 10.5...",NaN,NaN,NaN,None
XK005,NaN,NaN,NaN,NaN,NaN,95.86,NaN,"{'Wheat': 8.84, 'Barley': 0.33, 'Maize': 18.27...",NaN,NaN,NaN,None


In [18]:
lau_lookup_table = {}


for ctr in list(set((nuts3_df["CNTR_CODE"].values))):
    if ctr == "UK":
        pass
    else:
        lookup_df = pd.read_excel(path.join(utils.raw_data_dir, "EU-27-LAU-2023-NUTS-2021.xlsx"), sheet_name=ctr)
        break

TypeError: list indices must be integers or slices, not float

In [ ]:
eu_nats_lau_lookup

,Created:,2024-02-20 00:00:00
0,NaN,NaN
1,Updated on:,2024-04-22 00:00:00
2,Changes made:,"Revised DEGURBA and Coastal codes added to BE,..."
3,NaN,NaN
4,Updated on:,2024-04-23 00:00:00
...,...,...
78,Updated on:,2026-01-29 00:00:00
79,Changes made:,City Name: The term (greater city) has been ad...
80,NaN,NaN
81,Updated on:,2026-02-12 00:00:00
